# 6.2 Infrastructure monitor (run in SageMaker Studio)

CloudWatch alarms on SageMaker job utilisation (`/aws/sagemaker/TrainingJobs|TransformJobs|ProcessingJobs`: CPU, memory, disk) plus alarms on the custom `pamap2-har/Monitoring` metrics published by 6.0/6.1 (violation counts, degradation, job-failed 0/1). No SNS topic or EventBridge rule (user decision): alarm state is shown here and on the dashboard. Alarm creation is idempotent (`put_metric_alarm` overwrites). Job alarms are Metrics Insights query alarms over the whole job namespace (one query per job namespace and metric), so future jobs are watched too. Custom-metric alarms use `TreatMissingData=ignore` so the last evaluated state persists between manual runs.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import io, json, time
import boto3, numpy as np, pandas as pd
from botocore.exceptions import ClientError
from src.pamap2.config import load_config, s3_uri
from src.pamap2.schema import ACTIVITY_NAMES
from src.pamap2 import evaluation as E
from src.pamap2 import monitoring as M
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
print("Caller:", session.client("sts").get_caller_identity()["Arn"])
s3 = session.client("s3")
cw = session.client("cloudwatch")
sm = session.client("sagemaker")
SPLITS = s3_uri(cfg, "splits")
REPORTS = s3_uri(cfg, "models", "reports/")
MON = lambda key: s3_uri(cfg, "monitoring", key)
NS = cfg["monitoring_namespace"]

def split_key(uri):
    b, k = uri[len("s3://"):].split("/", 1)
    return b, k
def read_bytes(uri):
    b, k = split_key(uri)
    return s3.get_object(Bucket=b, Key=k)["Body"].read()
def read_bytes_or_none(uri):
    try:
        return read_bytes(uri)
    except ClientError as e:
        if e.response["Error"]["Code"] in ("NoSuchKey", "404"):
            return None
        raise
def read_parquet(name):
    return pd.read_parquet(io.BytesIO(read_bytes(SPLITS + name)))
def read_json(uri):
    return json.loads(read_bytes(uri))
def write_json(uri, obj):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=json.dumps(obj, indent=2).encode())
    print("Wrote", uri)
def put_text(uri, text):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=text.encode())
    return uri
def exists(uri):
    b, k = split_key(uri)
    try:
        s3.head_object(Bucket=b, Key=k)
        return True
    except ClientError:
        return False
def latest_subprefix(uri_prefix):
    """Newest child 'folder' (timestamped run folders sort lexicographically)."""
    b, k = split_key(uri_prefix)
    r = s3.list_objects_v2(Bucket=b, Prefix=k, Delimiter="/")
    subs = sorted(p["Prefix"] for p in r.get("CommonPrefixes", []))
    return f"s3://{b}/{subs[-1]}" if subs else None
def publish(metrics):
    """metrics: {name: value} -> custom namespace (put_metric_data)."""
    cw.put_metric_data(Namespace=NS, MetricData=[{"MetricName": n, "Value": float(v), "Unit": "None"} for n, v in metrics.items()])
    print("Published", metrics)

def safe_publish(metrics):
    try:
        publish(metrics)
    except Exception as e:   # never mask the original error
        print("WARNING: publish failed:", e)
def report_failure(job_name):
    try:
        print("FailureReason:", sm.describe_processing_job(ProcessingJobName=job_name).get("FailureReason"))
    except Exception as e:
        print("Could not describe job:", e)

FEATURES = read_json(SPLITS + "feature_columns.json")
LABEL_MAP = E.load_label_map(read_json(SPLITS + "label_map.json"))   # class index -> activity_id
K = len(LABEL_MAP)
ACTIVITY_IDS = list(LABEL_MAP.values())
print(K, "classes;", len(FEATURES), "features")


In [ ]:
def put_alarm(spec):
    """put_metric_alarm; if AccessDenied on actions, retry without AlarmActions (none are configured by default)."""
    try:
        cw.put_metric_alarm(**spec)
    except ClientError as e:
        if e.response["Error"]["Code"] != "AccessDenied" or "AlarmActions" not in spec:
            raise
        print("WARNING: AccessDenied for alarm actions; creating", spec["AlarmName"], "without actions")
        cw.put_metric_alarm(**{k: v for k, v in spec.items() if k != "AlarmActions"})


## Job-utilisation alarms

In [ ]:
prefix = cfg["monitoring_alarm_prefix"]
created = []
THRESHOLDS = {"CPUUtilization": 95, "MemoryUtilization": 90, "DiskUtilization": 85}
for job_type in M.JOB_NAMESPACES:
    for metric, thr in THRESHOLDS.items():
        spec = M.alarm_spec(job_type, metric, thr, prefix=prefix)   # Metrics Insights query: covers every current and future job
        put_alarm(spec); created.append(spec["AlarmName"])
print("Created/updated:", created)


## Monitoring-metric alarms (custom namespace)

In [ ]:
for metric, thr in {"DataJobFailed": 0, "ModelJobFailed": 0, "DataViolationCount": 0,
                    "ModelViolationCount": 0, "ModelDegraded": 0}.items():
    spec = M.alarm_spec("custom", metric, thr, prefix=prefix, namespace=NS, period=3600)
    put_alarm(spec); print("Created/updated", spec["AlarmName"])


## Alarm states

In [ ]:
alarms = cw.describe_alarms(AlarmNamePrefix=prefix + "-")["MetricAlarms"]
def _alarm_metric(a):
    # Metric-math alarms have no top-level MetricName; show their expression / metric names instead.
    if "MetricName" in a:
        return a["MetricName"]
    return ", ".join(m.get("Expression") or m.get("MetricStat", {}).get("Metric", {}).get("MetricName", m["Id"])
                     for m in a.get("Metrics", []))
display(pd.DataFrame([{"alarm": a["AlarmName"], "state": a["StateValue"], "metric": _alarm_metric(a)}
                      for a in alarms]))


## Cleanup

This notebook creates only alarms (kept on purpose, free tier). Verify nothing else is running.

In [ ]:
# Cleanup: nothing here creates schedules or endpoints; list them, delete only temporary pamap2-har-mon* models.
for page in sm.get_paginator("list_monitoring_schedules").paginate(NameContains=cfg["prefix"]):
    for x in page["MonitoringScheduleSummaries"]:
        print("WARNING: monitoring schedule exists (not deleted):", x["MonitoringScheduleName"])
for page in sm.get_paginator("list_models").paginate(NameContains=f"{cfg['prefix']}-mon"):
    for m in page["Models"]:
        if m["ModelName"].startswith(f"{cfg['prefix']}-mon"):
            print("Deleting temporary model", m["ModelName"])
            sm.delete_model(ModelName=m["ModelName"])
for kind, pager, key, field in (("processing", "list_processing_jobs", "ProcessingJobSummaries", "ProcessingJobName"),
                                ("transform", "list_transform_jobs", "TransformJobSummaries", "TransformJobName")):
    for page in sm.get_paginator(pager).paginate(StatusEquals="InProgress", NameContains=cfg["prefix"]):
        for x in page[key]:
            print("WARNING: still running", kind, "job:", x[field])
print("Endpoints:", [e["EndpointName"] for p in sm.get_paginator("list_endpoints").paginate() for e in p["Endpoints"]])
